# 🛡️ Toxic Comment Filtering & Moderation Alert System
### High-Fidelity Gaming Chat Moderation Pipeline

This notebook demonstrates a **multi-label toxicity classifier** designed to run in real-time on a gaming platform's live chat server. The pipeline automatically ingests messages, preprocesses them for gaming-specific slang and emojis, flags violations, computes a **0 to 5 severity score**, and triggers appropriate mitigation actions (**ALLOW**, **WARN**, **MUTE**, **BAN**).

### 🎯 Project Objectives
1. **Preprocessing Pipeline**: Handle casing, contractions, emoji conversions, and gaming-specific abbreviations (e.g. *stfu*, *kys*, *ez*).
2. **Multi-Label Toxicity Classifier**: Train a model that classifies comments into 6 categories: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, and `identity_hate` (hate speech).
3. **Precision Optimization**: Ensure **Precision > 80%** specifically for the `severe_toxic` class to minimize false-positive bans in competitive environments.
4. **Unified Scoring**: Design a mathematical severity score from 0 to 5.
5. **Dynamic Actions**: Map severity scores to specific moderation responses (Warn, Mute, Ban).
6. **Evaluation & Visualization**: Show metrics distributions, infraction heatmaps, and simulated live chat logs.

---  
## 🛠️ Step 1: Environment Setup & Package Verification

We begin by importing the necessary python libraries and verifying that PyTorch is configured with GPU/CUDA acceleration.

In [ ]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, hamming_loss, precision_recall_curve

# Add root directory to path to enable local imports
sys.path.append(os.getcwd())

import torch
print("Python Version:", sys.version.split()[0])
print("PyTorch Version:", torch.__version__)
print("CUDA GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

---  
## 🧹 Step 2: Gaming-Specific Text Preprocessing

Gaming chat is heavy with emojis, slang, and contractions. Standard NLP pipelines fail because words like `kys` (kill yourself) or emojis like `🖕` are ignored or treated as out-of-vocabulary. 

We test our custom `TextPreprocessor` which handles:
- Emoji demojization (e.g. `🔥` $\rightarrow$ `:fire:`)
- Contraction expansion (e.g. `i'm` $\rightarrow$ `i am`)
- Slang normalization (e.g. `stfu` $\rightarrow$ `shut the fuck up`, `ez` $\rightarrow$ `easy`)
- Casing and character sanitization

In [ ]:
from src.preprocessing import TextPreprocessor

preprocessor = TextPreprocessor()

# Test typical gaming chat messages
samples = [
    "GG! That was an ez win. Youre actually decent.",
    "stfu noob, go uninstall kys 🖕🔥",
    "im gonna hack u tonight w8 for it"
]

for s in samples:
    print(f"Original: '{s}'")
    print(f"Cleaned:  '{preprocessor.preprocess(s)}'\n")

---  
## 📥 Step 3: Data Ingestion & Fallback Pipeline

We load our training data. The system automatically attempts to fetch the classic **Jigsaw Toxic Comment Dataset** from Hugging Face. If offline, it seamlessly falls back to a high-fidelity synthetic gaming chat dataset (4,000 samples) with realistic multi-label infractions.

In [ ]:
from src.data_loader import load_toxicity_dataset

df, is_synthetic = load_toxicity_dataset()
print(f"Dataset generated? {is_synthetic}")
print(f"Dataframe Shape: {df.shape}")
print("\nSample records:")
display(df.head(4))

Let's check the distributions of our categories in the dataset.

In [ ]:
categories = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
counts = df[categories].sum()

plt.figure(figsize=(10, 5))
sns.set_theme(style="darkgrid")
sns.barplot(x=counts.index, y=counts.values, palette="flare")
plt.title("Infraction Count Distribution in Training Data")
plt.ylabel("Count")
plt.xlabel("Category")
plt.show()

---  
## 🧠 Step 4: Model Ingestion & Training

We split the dataset into Train (80%) and Validation (20%) sets. We then train our classical ML classifier which implements a TF-IDF character and word n-gram pipeline backed by a Multi-Output Logistic Regression model.

In [ ]:
# Preprocess all comments
print("Applying text preprocessor to comments...")
df['clean_comment'] = df['comment_text'].apply(preprocessor.preprocess)
df = df[df['clean_comment'] != ""]

X = df['clean_comment'].values
y = df[categories].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from src.model import ToxicityClassifier
clf = ToxicityClassifier(categories=categories)
clf.fit(X_train, y_train)

---  
## 🎯 Step 5: Precision Optimization for "Severe Toxic" Class

A key constraint in high-quality gaming moderation is preventing false positives: we do not want to ban someone who was engaging in harmless competitive banter. Therefore, we require **Precision > 80%** on the `severe_toxic` class.

We call `optimize_thresholds` to find a custom decision threshold that satisfies this constraint on validation data.

In [ ]:
# Current default thresholds before tuning
print("Thresholds before optimization:", clf.thresholds)

# Run optimization
best_th = clf.optimize_thresholds(
    X_val, y_val, target_precision=0.82, target_class='severe_toxic'
)

print("\nUpdated thresholds after optimization:", clf.thresholds)

---  
## 📊 Step 6: Validation Performance & Visualizations

We evaluate the multi-label predictions against our validation split using Hamming Loss and a comprehensive per-class classification report.

In [ ]:
# Predict using the calibrated thresholds
preds_val = clf.predict(X_val)

y_pred = np.zeros((len(X_val), len(categories)))
for i, p_dict in enumerate(preds_val):
    for j, col in enumerate(categories):
        y_pred[i, j] = p_dict[col]

# Compute general Hamming Loss (lower is better)
print(f"Hamming Loss: {hamming_loss(y_val, y_pred):.5f}\n")

# Classification report
print(classification_report(y_val, y_pred, target_names=categories))

Let's look at the label correlations to see how different forms of toxicity occur together in live gaming server chats.

In [ ]:
corr = pd.DataFrame(y_val, columns=categories).corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Toxicity Category Correlation Map")
plt.show()

---  
## ⚖️ Step 7: Severity Scoring & Mitigation Policies

We map our classification predictions to a **0 to 5 Severity Score** using weighted probability outputs. We then define our escalation policy thresholds:
- `Severity < 1.0`: **ALLOW** (safe chat)
- `Severity 1.0 – 2.5`: **WARN** (system warning sent to player)
- `Severity 2.5 – 4.0`: **MUTE** (redact obscene words, mute user chat for 5 minutes)
- `Severity 4.0 – 5.0`: **BAN** (terminate player session, alert administrators)

We also test our structural `SpamDetector` for rapid-fire comments and link flooding.

In [ ]:
from src.moderator import ContentModerator

# Initialize our full Moderator wrapper
moderator = ContentModerator(classifier=clf)

# Dynamic threshold settings
moderator.threshold_warn = 1.0
moderator.threshold_mute = 2.5
moderator.threshold_ban = 4.0

# Simulating chat stream
chat_messages = [
    ("Alpha_Gamer", "hey clean shot dude, nice teamwork!"),
    ("Spammer_X", "CHECK OUT FREE SKINS AT WWW.FREESKINS.NET NO SCAM!"),
    ("Rager_01", "fuck this laggy server and stfu you stupid idiot"),
    ("Violent_Player", "i am going to find where you live and end you in real life"),
    ("Severe_Hater", "kys immigrant trash get off our server")
]

print("⚡ SIMULATING MODERATION BOT CHAT ANALYSIS:\n")
for user, msg in chat_messages:
    report = moderator.analyze_message(msg, username=user)
    print(f"User: {report['username']}")
    print(f"  ├─ Text: '{report['original_text']}'")
    print(f"  ├─ Cleaned: '{report['preprocessed_text']}'")
    print(f"  ├─ Spam? {report['is_spam']} (Reason: {report['spam_reason']})")
    print(f"  ├─ Active Flags: { {k: v for k, v in report['flags'].items() if v == 1} }")
    print(f"  ├─ Severity Score (0-5): {report['severity_score']}")
    print(f"  ├─ RECOMMENDED ACTION: {report['action']} ({report['action_reason']})")
    print(f"  └─ Redacted Text: '{report['redacted_text']}'")
    print("-" * 70)

---  
## 💾 Step 8: Save Model Artifacts

Finally, we serialize our trained classical model pipeline so it can be loaded instantly by our production Streamlit dashboard.

In [ ]:
model_save_path = "models/toxicity_classifier.joblib"
clf.save(model_save_path)
print("Notebook execution successfully complete! The system is fully ready for deployment.")